<a href="https://colab.research.google.com/github/srimathy-R04/AutoMob-Mechanic-App/blob/main/Phishing_URL_Detection_Research.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
print("Phishing URL research setup started!")

Phishing URL research setup started!


In [5]:

import pandas as pd
import numpy as np
import sklearn
import xgboost

print("Pandas:", pd.__version__)
print("NumPy:", np.__version__)
print("Scikit-learn:", sklearn.__version__)
print("XGBoost:", xgboost.__version__)
print("Environment ready!")


Pandas: 2.2.3
NumPy: 2.1.3
Scikit-learn: 1.6.1
XGBoost: 3.4.1
Environment ready!


In [6]:

!pip -q install ucimlrepo

from ucimlrepo import fetch_ucirepo
import pandas as pd

# Download the official PhiUSIIL dataset
dataset = fetch_ucirepo(id=967)

# Combine input features and target label
X = dataset.data.features
y = dataset.data.targets

df = pd.concat([X, y], axis=1)

print("Dataset downloaded successfully!")
print("Dataset shape:", df.shape)
print("\nColumn names:")
print(df.columns.tolist())

print("\nFirst 5 rows:")
display(df.head())


Dataset downloaded successfully!
Dataset shape: (235795, 55)

Column names:
['URL', 'URLLength', 'Domain', 'DomainLength', 'IsDomainIP', 'TLD', 'URLSimilarityIndex', 'CharContinuationRate', 'TLDLegitimateProb', 'URLCharProb', 'TLDLength', 'NoOfSubDomain', 'HasObfuscation', 'NoOfObfuscatedChar', 'ObfuscationRatio', 'NoOfLettersInURL', 'LetterRatioInURL', 'NoOfDegitsInURL', 'DegitRatioInURL', 'NoOfEqualsInURL', 'NoOfQMarkInURL', 'NoOfAmpersandInURL', 'NoOfOtherSpecialCharsInURL', 'SpacialCharRatioInURL', 'IsHTTPS', 'LineOfCode', 'LargestLineLength', 'HasTitle', 'Title', 'DomainTitleMatchScore', 'URLTitleMatchScore', 'HasFavicon', 'Robots', 'IsResponsive', 'NoOfURLRedirect', 'NoOfSelfRedirect', 'HasDescription', 'NoOfPopup', 'NoOfiFrame', 'HasExternalFormSubmit', 'HasSocialNet', 'HasSubmitButton', 'HasHiddenFields', 'HasPasswordField', 'Bank', 'Pay', 'Crypto', 'HasCopyrightInfo', 'NoOfImage', 'NoOfCSS', 'NoOfJS', 'NoOfSelfRef', 'NoOfEmptyRef', 'NoOfExternalRef', 'label']

First 5 rows:


,URL,URLLength,Domain,DomainLength,IsDomainIP,TLD,URLSimilarityIndex,CharContinuationRate,TLDLegitimateProb,URLCharProb,...,Pay,Crypto,HasCopyrightInfo,NoOfImage,NoOfCSS,NoOfJS,NoOfSelfRef,NoOfEmptyRef,NoOfExternalRef,label
0,https://www.southbankmosaics.com,31,www.southbankmosaics.com,24,0,com,100.0,1.000000,0.522907,0.061933,...,0,0,1,34,20,28,119,0,124,1
1,https://www.uni-mainz.de,23,www.uni-mainz.de,16,0,de,100.0,0.666667,0.032650,0.050207,...,0,0,1,50,9,8,39,0,217,1
2,https://www.voicefmradio.co.uk,29,www.voicefmradio.co.uk,22,0,uk,100.0,0.866667,0.028555,0.064129,...,0,0,1,10,2,7,42,2,5,1
3,https://www.sfnmjournal.com,26,www.sfnmjournal.com,19,0,com,100.0,1.000000,0.522907,0.057606,...,1,1,1,3,27,15,22,1,31,1
4,https://www.rewildingargentina.org,33,www.rewildingargentina.org,26,0,org,100.0,1.000000,0.079963,0.059441,...,1,0,1,244,15,34,72,1,85,1


In [7]:
# Check the target labels
print("Label counts:")
print(df["label"].value_counts().sort_index())

print("\nLabel percentages:")
print((df["label"].value_counts(normalize=True) * 100).round(2).sort_index())

# Check missing values
print("\nTotal missing values:", df.isnull().sum().sum())

# Check duplicate rows
print("Duplicate rows:", df.duplicated().sum())

# Check data types
print("\nData types:")
print(df.dtypes.value_counts())

Label counts:
label
0    100945
1    134850
Name: count, dtype: int64

Label percentages:
label
0    42.81
1    57.19
Name: proportion, dtype: float64

Total missing values: 0
Duplicate rows: 0

Data types:
int64      41
float64    10
object      4
Name: count, dtype: int64


In [8]:
# Separate target label from input columns
target_column = "label"

# Display text columns
text_columns = df.select_dtypes(include=["object"]).columns.tolist()
print("Text columns:", text_columns)

# Display numeric columns
numeric_columns = df.select_dtypes(include=["number"]).columns.tolist()
print("\nNumber of numeric columns:", len(numeric_columns))
print("Numeric columns:", numeric_columns)

# Confirm target labels
print("\nTarget labels:", sorted(df[target_column].unique()))

Text columns: ['URL', 'Domain', 'TLD', 'Title']

Number of numeric columns: 51
Numeric columns: ['URLLength', 'DomainLength', 'IsDomainIP', 'URLSimilarityIndex', 'CharContinuationRate', 'TLDLegitimateProb', 'URLCharProb', 'TLDLength', 'NoOfSubDomain', 'HasObfuscation', 'NoOfObfuscatedChar', 'ObfuscationRatio', 'NoOfLettersInURL', 'LetterRatioInURL', 'NoOfDegitsInURL', 'DegitRatioInURL', 'NoOfEqualsInURL', 'NoOfQMarkInURL', 'NoOfAmpersandInURL', 'NoOfOtherSpecialCharsInURL', 'SpacialCharRatioInURL', 'IsHTTPS', 'LineOfCode', 'LargestLineLength', 'HasTitle', 'DomainTitleMatchScore', 'URLTitleMatchScore', 'HasFavicon', 'Robots', 'IsResponsive', 'NoOfURLRedirect', 'NoOfSelfRedirect', 'HasDescription', 'NoOfPopup', 'NoOfiFrame', 'HasExternalFormSubmit', 'HasSocialNet', 'HasSubmitButton', 'HasHiddenFields', 'HasPasswordField', 'Bank', 'Pay', 'Crypto', 'HasCopyrightInfo', 'NoOfImage', 'NoOfCSS', 'NoOfJS', 'NoOfSelfRef', 'NoOfEmptyRef', 'NoOfExternalRef', 'label']

Target labels: [np.int64(0), 

In [9]:
# Compare average feature values for each class
features_to_check = [
    "URLLength",
    "NoOfSubDomain",
    "HasObfuscation",
    "NoOfObfuscatedChar",
    "ObfuscationRatio",
    "IsHTTPS"
]

print("Feature averages by label:")
display(df.groupby("label")[features_to_check].mean().round(3))

Feature averages by label:


,URLLength,NoOfSubDomain,HasObfuscation,NoOfObfuscatedChar,ObfuscationRatio,IsHTTPS
label,,,,,,
0,45.720,1.169,0.005,0.058,0.0,0.492
1,26.229,1.162,0.000,0.000,0.0,1.000


In [10]:
df_phiusiil = df.copy()

print("Working dataset saved in memory.")
print("Rows:", len(df_phiusiil))
print("Columns:", len(df_phiusiil.columns))

Working dataset saved in memory.
Rows: 235795
Columns: 55


In [11]:
# Check whether the raw URL and related columns are available
columns_to_check = [
    "URL",
    "URLLength",
    "Domain",
    "DomainLength",
    "NoOfSubDomain",
    "IsDomainIP",
    "TLD",
    "label"
]

print("Required columns available:")
for col in columns_to_check:
    print(f"{col}: {col in df_phiusiil.columns}")

print("\nExample URLs:")
display(df_phiusiil[["URL", "Domain", "URLLength", "label"]].head(10))

Required columns available:
URL: True
URLLength: True
Domain: True
DomainLength: True
NoOfSubDomain: True
IsDomainIP: True
TLD: True
label: True

Example URLs:


,URL,Domain,URLLength,label
0,https://www.southbankmosaics.com,www.southbankmosaics.com,31,1
1,https://www.uni-mainz.de,www.uni-mainz.de,23,1
2,https://www.voicefmradio.co.uk,www.voicefmradio.co.uk,29,1
3,https://www.sfnmjournal.com,www.sfnmjournal.com,26,1
4,https://www.rewildingargentina.org,www.rewildingargentina.org,33,1
5,https://www.globalreporting.org,www.globalreporting.org,30,1
6,https://www.saffronart.com,www.saffronart.com,25,1
7,https://www.nerdscandy.com,www.nerdscandy.com,25,1
8,https://www.hyderabadonline.in,www.hyderabadonline.in,29,1
9,https://www.aap.org,www.aap.org,18,1


In [12]:
from urllib.parse import urlparse

def count_hyphens_in_path(url):
    path = urlparse(str(url)).path
    return path.count("-")

def count_dots_in_url(url):
    return str(url).count(".")

# Create experimental lexical features
df_phiusiil["qty_hyphen_path_exp"] = (
    df_phiusiil["URL"].apply(count_hyphens_in_path)
)

df_phiusiil["qty_dot_url_exp"] = (
    df_phiusiil["URL"].apply(count_dots_in_url)
)

print("Experimental lexical features created!")
display(
    df_phiusiil[
        ["URL", "qty_hyphen_path_exp", "qty_dot_url_exp"]
    ].head(10)
)

Experimental lexical features created!


,URL,qty_hyphen_path_exp,qty_dot_url_exp
0,https://www.southbankmosaics.com,0,2
1,https://www.uni-mainz.de,0,2
2,https://www.voicefmradio.co.uk,0,3
3,https://www.sfnmjournal.com,0,2
4,https://www.rewildingargentina.org,0,2
5,https://www.globalreporting.org,0,2
6,https://www.saffronart.com,0,2
7,https://www.nerdscandy.com,0,2
8,https://www.hyderabadonline.in,0,2
9,https://www.aap.org,0,2


In [13]:
# Summarize the experimental features
experimental_features = [
    "qty_hyphen_path_exp",
    "qty_dot_url_exp"
]

display(
    df_phiusiil.groupby("label")[experimental_features]
    .agg(["mean", "median", "max"])
    .round(3)
)

qty_hyphen_path_exp             qty_dot_url_exp           
                     mean median  max            mean median max
label                                                           
0                   0.128    0.0  286           2.385    2.0  99
1                   0.000    0.0    0           2.162    2.0   5

In [14]:
# Inspect URLs with the highest experimental feature values
print("URLs with the most hyphens in their paths:")
display(
    df_phiusiil.nlargest(5, "qty_hyphen_path_exp")[
        ["URL", "qty_hyphen_path_exp", "label"]
    ]
)

print("\nURLs with the most dots:")
display(
    df_phiusiil.nlargest(5, "qty_dot_url_exp")[
        ["URL", "qty_dot_url_exp", "label"]
    ]
)

URLs with the most hyphens in their paths:


,URL,qty_hyphen_path_exp,label
150870,https://bodhicardstech.com/beta/wx%20qz%20xx%2...,286,0
152736,https://bodhicardstech.com/beta/wx%20qz%20xx%2...,269,0
145392,https://inspirationalatmospheres.com/qxzrrr%20...,250,0
146920,http://www.inspirationalatmospheres.com/qxzrrr...,232,0
157284,https://inspirationalatmospheres.com/qxzrrr%20...,232,0



URLs with the most dots:


,URL,qty_dot_url_exp,label
150870,https://bodhicardstech.com/beta/wx%20qz%20xx%2...,99,0
152736,https://bodhicardstech.com/beta/wx%20qz%20xx%2...,93,0
145392,https://inspirationalatmospheres.com/qxzrrr%20...,90,0
146920,http://www.inspirationalatmospheres.com/qxzrrr...,85,0
157284,https://inspirationalatmospheres.com/qxzrrr%20...,84,0


In [15]:
# Inspect the URL with the largest feature values
sample_url = df_phiusiil.loc[150870, "URL"]

print("Full URL:")
print(sample_url)

print("\nTotal URL characters:", len(sample_url))
print("Hyphens in parsed path:", df_phiusiil.loc[150870, "qty_hyphen_path_exp"])
print("Dots in URL:", df_phiusiil.loc[150870, "qty_dot_url_exp"])

print("\nLabel:", df_phiusiil.loc[150870, "label"])

Full URL:
https://bodhicardstech.com/beta/wx%20qz%20xx%20ww%20qxz%20wes%20wsed/54dr%2056yg%2065yft5%204f6yft5%20d46yg5%20fyt45t%20f4d4%20yf56td4/ft5%20fy%2065y%204%205tdr34%204yf46%20y5uyf%20t5%20d6yf5%20td%204fdtr34%205yf5%2064d5t34f46u75/gy6f5t%2056yf%205t%204g75%20fy65t%204%2065gf%20y%2054f%204g5ug7%20u5y6f4%205%20guf%20y6545d%204f6gu%207f65y%204dt/gy65ft%2045%20fy6%206g57guf%20y5%20f4%205%20y57%20yf6%204t5%20gu%207f5y645%20%20fu%20f5y%2064dt4tf/g4gr%20y65f%20f5yft4%2056yf%2045t%205gy%20f645ft%20475yf6%204tf5%20yft%2065ug7%2056fy45t%204%20u5fy6t%20f4u%207ug5y%204tf465%20u%206ug7y655/y65t%2056gy%2056gu6y%2045%20g7u%20y6f4%205g6u7u5%206yf4%2075gu6%20yf4%206gy5%20ft46u%207i6u7g5%20yf45%2056gu%20ug76%20y566f4%205u%2068%2067ug6y%20f4g4%2076ugy5%2045%20u76guy54/f54_4yg=u-y65f5t4t.46gu-g5=y64f-u5g7y6_46gu=-u6g75y6.3dt-46=u68gu-75_57hi6=8i-g6u7fyd4.u6-8i=6u56-yf45-46g=u57-86ug7f5y6d4.yf5-7u=86-u756y_46=g57h-8i6gu75.34rs-d5=y56-gu-6h=87i-h86gu756y.45yf-56u=68-gu75y-56gu6gu75y6/y65ft3d4_5fy=6

In [16]:
# Exploratory check only—not the paper's verified feature
brand_keywords = [
    "paypal", "microsoft", "google", "apple",
    "amazon", "facebook", "netflix", "instagram"
]

def contains_brand_keyword(url):
    url = str(url).lower()
    return int(any(brand in url for brand in brand_keywords))

df_phiusiil["has_brand_keyword_exp"] = (
    df_phiusiil["URL"].apply(contains_brand_keyword)
)

print("Experimental brand-keyword counts by label:")
display(
    df_phiusiil.groupby("label")["has_brand_keyword_exp"]
    .agg(["sum", "mean", "count"])
    .round(4)
)

Experimental brand-keyword counts by label:


,sum,mean,count
label,,,
0,3109,0.0308,100945
1,92,0.0007,134850


In [17]:
# Show 10 phishing URLs containing a brand keyword
display(
    df_phiusiil.loc[
        (df_phiusiil["label"] == 0) &
        (df_phiusiil["has_brand_keyword_exp"] == 1),
        ["URL", "label", "has_brand_keyword_exp"]
    ].head(10)
)

,URL,label,has_brand_keyword_exp
77,https://s3.amazonaws.com/appforest_uf/f1678949...,0,1
266,http://www.googleimg.top,0,1
286,http://www.pony4.microsoftups.com,0,1
356,https://objectstorage.eu-frankfurt-1.oracleclo...,0,1
376,http://www.amazon.co.jp.infotoop.shop/,0,1
401,http://43.134.167.94/servicelogin?passive=1209...,0,1
402,https://docs.google.com/forms/d/1casphevkfdaz2...,0,1
522,http://www.alma.apple.cloudns.org,0,1
528,https://bacthanh0204.github.io/dangnhapfacebook/,0,1
661,http://www.helpmicrosoft.net,0,1


In [18]:
# Show legitimate URLs containing a brand keyword
display(
    df_phiusiil.loc[
        (df_phiusiil["label"] == 1) &
        (df_phiusiil["has_brand_keyword_exp"] == 1),
        ["URL", "label", "has_brand_keyword_exp"]
    ].head(10)
)

,URL,label,has_brand_keyword_exp
176,https://www.wp-amazon-plugin.com,1,1
999,https://www.applethoughts.com,1,1
2094,https://www.apple.com.cn,1,1
2334,https://www.appledaily.com.tw,1,1
2465,https://www.arcticapples.com,1,1
2578,https://www.buyappleply.com,1,1
2602,https://www.delightfuldapple.com,1,1
2859,https://www.applescrapple.com,1,1
3958,https://www.mirrors.apple2.org.za,1,1
5621,https://www.open-apple.net,1,1


In [19]:
from urllib.parse import urlparse
import re

brand_keywords = [
    "paypal", "microsoft", "google", "apple",
    "amazon", "facebook", "netflix", "instagram"
]

def hostname_brand_token(url):
    hostname = urlparse(str(url)).hostname or ""
    tokens = re.split(r"[^a-z0-9]+", hostname.lower())

    return int(any(brand in tokens for brand in brand_keywords))

df_phiusiil["has_brand_hostname_token_exp"] = (
    df_phiusiil["URL"].apply(hostname_brand_token)
)

comparison = df_phiusiil.groupby("label").agg(
    original_feature=("has_brand_keyword_exp", "mean"),
    hostname_token_feature=("has_brand_hostname_token_exp", "mean"),
    total_urls=("URL", "count")
)

comparison[["original_feature", "hostname_token_feature"]] *= 100

display(comparison.round(3))

,original_feature,hostname_token_feature,total_urls
label,,,
0,3.080,1.387,100945
1,0.068,0.016,134850


In [20]:
from sklearn.metrics import precision_score, recall_score

y_true = df_phiusiil["label"]
y_phishing = (y_true == 0).astype(int)

for feature in [
    "has_brand_keyword_exp",
    "has_brand_hostname_token_exp"
]:
    predictions = df_phiusiil[feature]

    print(f"\nFeature: {feature}")
    print("Phishing detection rate:",
          round(recall_score(y_phishing, predictions) * 100, 3), "%")
    print("Legitimate URLs flagged:",
          round(predictions[y_true == 1].mean() * 100, 3), "%")
    print("Precision:",
          round(precision_score(y_phishing, predictions, zero_division=0) * 100, 3), "%")


Feature: has_brand_keyword_exp
Phishing detection rate: 3.08 %
Legitimate URLs flagged: 0.068 %
Precision: 97.126 %

Feature: has_brand_hostname_token_exp
Phishing detection rate: 1.387 %
Legitimate URLs flagged: 0.016 %
Precision: 98.522 %


prepare the baseline dataset

In [21]:

# Step 1: Prepare the original dataset for baseline training

df_baseline = df.copy()

# Separate input features (X) and target label (y)
X = df_baseline.drop(
    columns=["label", "URL", "Domain", "TLD", "Title"]
)

y = df_baseline["label"]

print("Dataset shape:", df_baseline.shape)
print("Input feature shape:", X.shape)
print("Target shape:", y.shape)

print("\nClass distribution:")
print(y.value_counts().sort_index())

print("\nNon-numeric input columns:")
print(X.select_dtypes(exclude="number").columns.tolist())

print("\nMissing values in X:", X.isnull().sum().sum())


Dataset shape: (235795, 55)
Input feature shape: (235795, 50)
Target shape: (235795,)

Class distribution:
label
0    100945
1    134850
Name: count, dtype: int64

Non-numeric input columns:
[]

Missing values in X: 0


In [22]:

from sklearn.model_selection import train_test_split

# Split the dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training features:", X_train.shape)
print("Testing features:", X_test.shape)

print("\nTraining label distribution:")
print(y_train.value_counts(normalize=True).sort_index().round(4))

print("\nTesting label distribution:")
print(y_test.value_counts(normalize=True).sort_index().round(4))


Training features: (188636, 50)
Testing features: (47159, 50)

Training label distribution:
label
0    0.4281
1    0.5719
Name: proportion, dtype: float64

Testing label distribution:
label
0    0.4281
1    0.5719
Name: proportion, dtype: float64


In [23]:

from xgboost import XGBClassifier

# Create the baseline XGBoost model
xgb_model = XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    random_state=42,
    eval_metric="logloss",
    n_jobs=-1
)

# Train the model using training data
xgb_model.fit(X_train, y_train)

print("XGBoost baseline model training completed!")


XGBoost baseline model training completed!


In [24]:

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

# Predict labels for unseen test data
y_pred = xgb_model.predict(X_test)

# Calculate evaluation metrics
print("XGBoost Baseline Results")
print("-" * 35)

print("Accuracy :", round(accuracy_score(y_test, y_pred) * 100, 2), "%")
print("Precision:", round(precision_score(y_test, y_pred, pos_label=0) * 100, 2), "%")
print("Recall   :", round(recall_score(y_test, y_pred, pos_label=0) * 100, 2), "%")
print("F1-score :", round(f1_score(y_test, y_pred, pos_label=0) * 100, 2), "%")

print("\nClassification Report:")
print(classification_report(
    y_test,
    y_pred,
    target_names=["Phishing", "Legitimate"],
    digits=4
))

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))


XGBoost Baseline Results
-----------------------------------
Accuracy : 100.0 %
Precision: 100.0 %
Recall   : 100.0 %
F1-score : 100.0 %

Classification Report:
              precision    recall  f1-score   support

    Phishing     1.0000    1.0000    1.0000     20189
  Legitimate     1.0000    1.0000    1.0000     26970

    accuracy                         1.0000     47159
   macro avg     1.0000    1.0000    1.0000     47159
weighted avg     1.0000    1.0000    1.0000     47159

Confusion Matrix:
[[20189     0]
 [    0 26970]]


In [25]:

import pandas as pd

# 1. Find the features the model relies on most
importance = pd.Series(
    xgb_model.feature_importances_,
    index=X_train.columns
).sort_values(ascending=False)

print("Top 15 most important features:")
display(importance.head(15).to_frame("importance").round(4))

# 2. Check whether any feature is strongly related to the label
correlations = (
    X_train.corrwith(y_train)
    .abs()
    .sort_values(ascending=False)
)

print("\nTop 15 features by absolute correlation with label:")
display(correlations.head(15).to_frame("abs_correlation").round(4))

# 3. Check whether identical input rows occur in both splits
train_rows = set(
    pd.util.hash_pandas_object(X_train, index=False).tolist()
)
test_rows = set(
    pd.util.hash_pandas_object(X_test, index=False).tolist()
)

print("\nUnique training input rows:", len(train_rows))
print("Unique testing input rows:", len(test_rows))
print("Input rows appearing in both splits:", len(train_rows & test_rows))


Top 15 most important features:


,importance
URLSimilarityIndex,0.9798
IsHTTPS,0.0097
LineOfCode,0.0094
NoOfSelfRef,0.0002
HasSocialNet,0.0001
NoOfExternalRef,0.0001
LargestLineLength,0.0001
NoOfImage,0.0001
NoOfLettersInURL,0.0001
NoOfJS,0.0001



Top 15 features by absolute correlation with label:


,abs_correlation
URLSimilarityIndex,0.8604
HasSocialNet,0.7837
HasCopyrightInfo,0.7428
HasDescription,0.6906
IsHTTPS,0.6091
DomainTitleMatchScore,0.5835
HasSubmitButton,0.5790
IsResponsive,0.5485
URLTitleMatchScore,0.5384
SpacialCharRatioInURL,0.5337



Unique training input rows: 188054
Unique testing input rows: 47101
Input rows appearing in both splits: 168


In [26]:

# Examine URLSimilarityIndex by class label

display(
    df_baseline.groupby("label")["URLSimilarityIndex"]
    .agg(["count", "min", "median", "mean", "max"])
    .round(4)
)

print("\nMost common values by label:")
for label in [0, 1]:
    print(f"\nLabel {label}:")
    print(
        df_baseline.loc[
            df_baseline["label"] == label,
            "URLSimilarityIndex"
        ].value_counts().head(10)
    )


,count,min,median,mean,max
label,,,,,
0,100945,0.1556,51.4193,49.617,100.0
1,134850,100.0000,100.0000,100.000,100.0



Most common values by label:

Label 0:
URLSimilarityIndex
100.000000    786
66.720000     743
66.695385     498
36.681914     488
22.334010     369
49.385610     363
60.845588     237
80.000000     220
9.540762      216
23.030879     201
Name: count, dtype: int64

Label 1:
URLSimilarityIndex
100.0    134850
Name: count, dtype: int64


In [27]:

from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix
)

# Remove the suspiciously dominant feature
X_no_similarity = X.drop(columns=["URLSimilarityIndex"])

# Keep the same reproducible 80/20 stratified split
X_train_ns, X_test_ns, y_train_ns, y_test_ns = train_test_split(
    X_no_similarity,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# Train a new model
xgb_no_similarity = XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    random_state=42,
    eval_metric="logloss",
    n_jobs=-1
)

xgb_no_similarity.fit(X_train_ns, y_train_ns)

# Evaluate on the test set
y_pred_ns = xgb_no_similarity.predict(X_test_ns)

print("XGBoost WITHOUT URLSimilarityIndex")
print("Accuracy :", round(accuracy_score(y_test_ns, y_pred_ns) * 100, 2), "%")
print("Precision:", round(precision_score(y_test_ns, y_pred_ns, pos_label=0) * 100, 2), "%")
print("Recall   :", round(recall_score(y_test_ns, y_pred_ns, pos_label=0) * 100, 2), "%")
print("F1-score :", round(f1_score(y_test_ns, y_pred_ns, pos_label=0) * 100, 2), "%")
print("\nConfusion matrix:")
print(confusion_matrix(y_test_ns, y_pred_ns))


XGBoost WITHOUT URLSimilarityIndex
Accuracy : 100.0 %
Precision: 100.0 %
Recall   : 100.0 %
F1-score : 100.0 %

Confusion matrix:
[[20188     1]
 [    0 26970]]


In [28]:
from urllib.parse import urlparse
import re
import pandas as pd

def extract_url_features(url):
    url = str(url)
    parsed = urlparse(url if "://" in url else "http://" + url)

    hostname = parsed.hostname or ""
    path = parsed.path or ""
    query = parsed.query or ""

    return {
        "url_length": len(url),
        "hostname_length": len(hostname),
        "path_length": len(path),
        "count_dots": url.count("."),
        "count_hyphens": url.count("-"),
        "count_digits": sum(c.isdigit() for c in url),
        "count_at": url.count("@"),
        "count_question": url.count("?"),
        "count_equal": url.count("="),
        "count_ampersand": url.count("&"),
        "count_percent": url.count("%"),
        "count_slashes": url.count("/"),
        "count_subdomains": max(0, len(hostname.split(".")) - 2),
        "has_https": int(parsed.scheme.lower() == "https"),
        "has_ip_in_host": int(
            bool(re.fullmatch(r"\d{1,3}(?:\.\d{1,3}){3}", hostname))
        ),
        "path_has_hyphen": int("-" in path),
        "query_present": int(bool(query))
    }

# Build a new feature table using only the raw URL string
url_features = pd.DataFrame(
    df["URL"].apply(extract_url_features).tolist(),
    index=df.index
)

y_url = df["label"]

print("Raw URL feature shape:", url_features.shape)
print(url_features.head())

Raw URL feature shape: (235795, 17)
   url_length  hostname_length  path_length  count_dots  count_hyphens  \
0          32               24            0           2              0   
1          24               16            0           2              1   
2          30               22            0           3              0   
3          27               19            0           2              0   
4          34               26            0           2              0   

   count_digits  count_at  count_question  count_equal  count_ampersand  \
0             0         0               0            0                0   
1             0         0               0            0                0   
2             0         0               0            0                0   
3             0         0               0            0                0   
4             0         0               0            0                0   

   count_percent  count_slashes  count_subdomains  has_https  has_ip

In [29]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)
from xgboost import XGBClassifier

# Split the raw URL features
X_train_url, X_test_url, y_train_url, y_test_url = train_test_split(
    url_features,
    y_url,
    test_size=0.20,
    random_state=42,
    stratify=y_url
)

# Create and train XGBoost
xgb_url_model = XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    random_state=42,
    eval_metric="logloss",
    n_jobs=-1
)

xgb_url_model.fit(X_train_url, y_train_url)

# Predict
y_pred_url = xgb_url_model.predict(X_test_url)

# Evaluate
print("XGBoost - RAW URL FEATURES ONLY")
print("Accuracy :", round(accuracy_score(y_test_url, y_pred_url) * 100, 2), "%")
print("Precision:", round(precision_score(y_test_url, y_pred_url, pos_label=0) * 100, 2), "%")
print("Recall   :", round(recall_score(y_test_url, y_pred_url, pos_label=0) * 100, 2), "%")
print("F1-score :", round(f1_score(y_test_url, y_pred_url, pos_label=0) * 100, 2), "%")

print("\nConfusion Matrix:")
print(confusion_matrix(y_test_url, y_pred_url))

print("\nClassification Report:")
print(classification_report(
    y_test_url,
    y_pred_url,
    labels=[0, 1],
    target_names=["Phishing", "Legitimate"],
    digits=4
))

XGBoost - RAW URL FEATURES ONLY
Accuracy : 99.53 %
Precision: 99.9 %
Recall   : 99.0 %
F1-score : 99.45 %

Confusion Matrix:
[[19987   202]
 [   21 26949]]

Classification Report:
              precision    recall  f1-score   support

    Phishing     0.9990    0.9900    0.9945     20189
  Legitimate     0.9926    0.9992    0.9959     26970

    accuracy                         0.9953     47159
   macro avg     0.9958    0.9946    0.9952     47159
weighted avg     0.9953    0.9953    0.9953     47159



In [30]:
from urllib.parse import urlparse
import pandas as pd

def get_hostname(url):
    parsed = urlparse(
        str(url) if "://" in str(url) else "http://" + str(url)
    )
    return (parsed.hostname or "").lower().rstrip(".")

# Extract domain groups from the original URLs
url_domains = df["URL"].apply(get_hostname)

print("Total URLs:", len(url_domains))
print("Unique hostnames:", url_domains.nunique())
print("URLs with missing hostname:", (url_domains == "").sum())
print("\nMost frequent hostnames:")
print(url_domains.value_counts().head(10))

Total URLs: 235795
Unique hostnames: 220077
URLs with missing hostname: 0

Most frequent hostnames:
URL
ipfs.io                         1197
docs.google.com                  526
mail.deliverylifesupport.com     370
cloudflare-ipfs.com              359
storageapi.fleek.co              318
gateway.pinata.cloud             275
gateway.ipfs.io                  249
ipfs.litnet.work                 223
cf-ipfs.com                      190
s3.amazonaws.com                 182
Name: count, dtype: int64


In [31]:
from sklearn.model_selection import GroupShuffleSplit
from xgboost import XGBClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, classification_report
)

# Split by hostname so the same hostname cannot appear in both sets
group_split = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    group_split.split(url_features, y_url, groups=url_domains)
)

X_train_group = url_features.iloc[train_idx]
X_test_group = url_features.iloc[test_idx]
y_train_group = y_url.iloc[train_idx]
y_test_group = y_url.iloc[test_idx]

# Verify that no hostname overlaps
train_domains = set(url_domains.iloc[train_idx])
test_domains = set(url_domains.iloc[test_idx])

print("Training samples:", len(train_idx))
print("Testing samples:", len(test_idx))
print("Training unique hostnames:", len(train_domains))
print("Testing unique hostnames:", len(test_domains))
print("Overlapping hostnames:", len(train_domains & test_domains))

# Train XGBoost on training domains
xgb_group_model = XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    random_state=42,
    eval_metric="logloss",
    n_jobs=-1
)

xgb_group_model.fit(X_train_group, y_train_group)

# Evaluate on unseen hostnames
y_pred_group = xgb_group_model.predict(X_test_group)

print("\nXGBoost - HOSTNAME-BASED SPLIT")
print("Accuracy :", round(accuracy_score(y_test_group, y_pred_group) * 100, 2), "%")
print("Precision:", round(precision_score(
    y_test_group, y_pred_group, pos_label=0, zero_division=0
) * 100, 2), "%")
print("Recall   :", round(recall_score(
    y_test_group, y_pred_group, pos_label=0, zero_division=0
) * 100, 2), "%")
print("F1-score :", round(f1_score(
    y_test_group, y_pred_group, pos_label=0, zero_division=0
) * 100, 2), "%")

print("\nConfusion matrix:")
print(confusion_matrix(y_test_group, y_pred_group, labels=[0, 1]))

print("\nClassification report:")
print(classification_report(
    y_test_group, y_pred_group,
    labels=[0, 1],
    target_names=["Phishing", "Legitimate"],
    digits=4,
    zero_division=0
))

Training samples: 188212
Testing samples: 47583
Training unique hostnames: 176061
Testing unique hostnames: 44016
Overlapping hostnames: 0

XGBoost - HOSTNAME-BASED SPLIT
Accuracy : 99.6 %
Precision: 99.9 %
Recall   : 99.18 %
F1-score : 99.54 %

Confusion matrix:
[[20393   168]
 [   20 27002]]

Classification report:
              precision    recall  f1-score   support

    Phishing     0.9990    0.9918    0.9954     20561
  Legitimate     0.9938    0.9993    0.9965     27022

    accuracy                         0.9960     47583
   macro avg     0.9964    0.9955    0.9960     47583
weighted avg     0.9961    0.9960    0.9960     47583



In [32]:
import pandas as pd

importance_df = pd.DataFrame({
    "Feature": url_features.columns,
    "Importance": xgb_group_model.feature_importances_
})

importance_df = importance_df.sort_values(
    by="Importance",
    ascending=False
).reset_index(drop=True)

print("Feature Importance — Hostname-Based Model")
print(importance_df.to_string(index=False))

Feature Importance — Hostname-Based Model
         Feature  Importance
       has_https    0.524831
     path_length    0.469347
      count_dots    0.002128
      url_length    0.001925
    count_digits    0.000890
   count_hyphens    0.000741
count_subdomains    0.000092
 hostname_length    0.000047
        count_at    0.000000
     count_equal    0.000000
  count_question    0.000000
   count_percent    0.000000
 count_ampersand    0.000000
   count_slashes    0.000000
  has_ip_in_host    0.000000
 path_has_hyphen    0.000000
   query_present    0.000000


In [33]:
from xgboost import XGBClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

# Remove the two dominant features
features_without_top2 = url_features.drop(
    columns=["has_https", "path_length"]
)

# Keep the same hostname-based train/test split
X_train_ablation = features_without_top2.iloc[train_idx]
X_test_ablation = features_without_top2.iloc[test_idx]

xgb_ablation = XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    random_state=42,
    eval_metric="logloss",
    n_jobs=-1
)

xgb_ablation.fit(X_train_ablation, y_train_group)

y_pred_ablation = xgb_ablation.predict(X_test_ablation)

print("XGBoost WITHOUT has_https and path_length")
print("Accuracy :", round(
    accuracy_score(y_test_group, y_pred_ablation) * 100, 2
), "%")
print("Precision:", round(
    precision_score(y_test_group, y_pred_ablation, pos_label=0,
                    zero_division=0) * 100, 2
), "%")
print("Recall   :", round(
    recall_score(y_test_group, y_pred_ablation, pos_label=0,
                 zero_division=0) * 100, 2
), "%")
print("F1-score :", round(
    f1_score(y_test_group, y_pred_ablation, pos_label=0,
             zero_division=0) * 100, 2
), "%")

print("\nConfusion matrix:")
print(confusion_matrix(
    y_test_group, y_pred_ablation, labels=[0, 1]
))

XGBoost WITHOUT has_https and path_length
Accuracy : 99.5 %
Precision: 99.8 %
Recall   : 99.03 %
F1-score : 99.41 %

Confusion matrix:
[[20362   199]
 [   41 26981]]


In [34]:
import pandas as pd

https_analysis = pd.crosstab(
    df["label"],
    url_features["has_https"],
    normalize="index"
) * 100

https_analysis.index = ["Phishing (0)", "Legitimate (1)"]
https_analysis.columns = ["HTTP", "HTTPS"]

print("HTTPS distribution within each class (%)")
print(https_analysis.round(2))

print("\nRaw counts:")
print(pd.crosstab(
    df["label"],
    url_features["has_https"]
))

HTTPS distribution within each class (%)
                 HTTP   HTTPS
Phishing (0)    51.26   48.74
Legitimate (1)   0.00  100.00

Raw counts:
has_https      0       1
label                   
0          51749   49196
1              0  134850


In [35]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

# Use HTTPS alone as the prediction:
# has_https = 0 -> predict phishing (0)
# has_https = 1 -> predict legitimate (1)

y_pred_https = url_features["has_https"]

print("HTTPS-ONLY BASELINE")
print("Accuracy :", round(
    accuracy_score(y_url, y_pred_https) * 100, 2
), "%")
print("Precision:", round(
    precision_score(y_url, y_pred_https, pos_label=0) * 100, 2
), "%")
print("Recall   :", round(
    recall_score(y_url, y_pred_https, pos_label=0) * 100, 2
), "%")
print("F1-score :", round(
    f1_score(y_url, y_pred_https, pos_label=0) * 100, 2
), "%")

print("\nConfusion matrix:")
print(confusion_matrix(y_url, y_pred_https, labels=[0, 1]))

HTTPS-ONLY BASELINE
Accuracy : 79.14 %
Precision: 100.0 %
Recall   : 51.26 %
F1-score : 67.78 %

Confusion matrix:
[[ 51749  49196]
 [     0 134850]]


In [36]:
# Compare URL path length by class
path_analysis = pd.DataFrame({
    "path_length": url_features["path_length"],
    "label": y_url
})

print("Path length summary by class:")
print(
    path_analysis.groupby("label")["path_length"]
    .describe()
    .round(2)
)

print("\nMedian path length by class:")
print(
    path_analysis.groupby("label")["path_length"]
    .median()
)

Path length summary by class:
          count  mean    std  min  25%  50%  75%     max
label                                                   
0      100945.0  8.73  35.31  0.0  0.0  1.0  6.0  4248.0
1      134850.0  0.00   0.00  0.0  0.0  0.0  0.0     0.0

Median path length by class:
label
0    1.0
1    0.0
Name: path_length, dtype: float64


In [37]:
# Inspect original URLs and extracted features
inspection = pd.DataFrame({
    "URL": df["URL"],
    "Label": df["label"],
    "HTTPS": url_features["has_https"],
    "Path_Length": url_features["path_length"]
})

print("LEGITIMATE URL EXAMPLES")
display(
    inspection[inspection["Label"] == 1][
        ["URL", "Label", "HTTPS", "Path_Length"]
    ].head(10)
)

print("\nPHISHING URL EXAMPLES")
display(
    inspection[inspection["Label"] == 0][
        ["URL", "Label", "HTTPS", "Path_Length"]
    ].head(10)
)

LEGITIMATE URL EXAMPLES


,URL,Label,HTTPS,Path_Length
0,https://www.southbankmosaics.com,1,1,0
1,https://www.uni-mainz.de,1,1,0
2,https://www.voicefmradio.co.uk,1,1,0
3,https://www.sfnmjournal.com,1,1,0
4,https://www.rewildingargentina.org,1,1,0
5,https://www.globalreporting.org,1,1,0
6,https://www.saffronart.com,1,1,0
7,https://www.nerdscandy.com,1,1,0
8,https://www.hyderabadonline.in,1,1,0
9,https://www.aap.org,1,1,0



PHISHING URL EXAMPLES


,URL,Label,HTTPS,Path_Length
11,http://www.teramill.com,0,0,0
20,http://www.f0519141.xsph.ru,0,0,0
21,http://www.shprakserf.gq,0,0,0
27,https://service-mitld.firebaseapp.com/,0,1,1
28,http://www.kuradox92.lima-city.de,0,0,0
29,https://liuy-9a930.web.app/,0,1,1
31,https://ipfs.io/ipfs/qmrvvyr84esa2assw9vvwupqj...,0,1,52
32,http://att-103731-107123.weeblysite.com/,0,0,1
34,https://hidok4f8zl.firebaseapp.com/,0,1,1
37,http://www.ooguy.com,0,0,0


In [38]:
# Count URLs with an empty versus nonempty path
path_check = pd.crosstab(
    y_url,
    url_features["path_length"] > 0
)

path_check.index = ["Phishing (0)", "Legitimate (1)"]
path_check.columns = ["Empty path", "Nonempty path"]

print(path_check)

print("\nPercentage with nonempty paths:")
print(
    (path_check["Nonempty path"] /
     path_check.sum(axis=1) * 100).round(2)
)

                Empty path  Nonempty path
Phishing (0)         40518          60427
Legitimate (1)      134850              0

Percentage with nonempty paths:
Phishing (0)      59.86
Legitimate (1)     0.00
dtype: float64


In [39]:
# Compare original URL length with the dataset's URLLength feature
url_length_check = pd.DataFrame({
    "Original_URL": df["URL"].astype(str),
    "Actual_String_Length": df["URL"].astype(str).str.len(),
    "Dataset_URLLength": df["URLLength"],
    "Label": df["label"]
})

url_length_check["Length_Matches"] = (
    url_length_check["Actual_String_Length"]
    == url_length_check["Dataset_URLLength"]
)

print("Length agreement:")
print(url_length_check["Length_Matches"].value_counts())

print("\nMismatch percentage:")
print(round(
    (~url_length_check["Length_Matches"]).mean() * 100, 2
), "%")

print("\nSample mismatches:")
display(
    url_length_check[
        ~url_length_check["Length_Matches"]
    ].head(10)
)

Length agreement:
Length_Matches
False    187151
True      48644
Name: count, dtype: int64

Mismatch percentage:
79.37 %

Sample mismatches:


,Original_URL,Actual_String_Length,Dataset_URLLength,Label,Length_Matches
0,https://www.southbankmosaics.com,32,31,1,False
1,https://www.uni-mainz.de,24,23,1,False
2,https://www.voicefmradio.co.uk,30,29,1,False
3,https://www.sfnmjournal.com,27,26,1,False
4,https://www.rewildingargentina.org,34,33,1,False
5,https://www.globalreporting.org,31,30,1,False
6,https://www.saffronart.com,26,25,1,False
7,https://www.nerdscandy.com,26,25,1,False
8,https://www.hyderabadonline.in,30,29,1,False
9,https://www.aap.org,19,18,1,False


In [40]:
# Calculate the difference between actual and stored URL lengths
length_difference = (
    df["URL"].astype(str).str.len() - df["URLLength"]
)

print("Length difference counts:")
print(length_difference.value_counts().sort_index().head(20))

print("\nPercentage where actual length is one character longer:")
print(round((length_difference == 1).mean() * 100, 2), "%")

print("\nExamples where difference is not 1:")
display(
    df.loc[
        length_difference != 1,
        ["URL", "URLLength"]
    ].head(10)
)

Length difference counts:
0      48644
1     187149
4          1
35         1
Name: count, dtype: int64

Percentage where actual length is one character longer:
79.37 %

Examples where difference is not 1:


,URL,URLLength
21,http://www.shprakserf.gq,24
32,http://att-103731-107123.weeblysite.com/,40
34,https://hidok4f8zl.firebaseapp.com/,35
45,https://pontosapontamentolu.com/gclid/=/c/?gcl...,140
52,https://sabadonoappmagazine.com/,32
73,https://ggsexpole.web.app/,26
77,https://s3.amazonaws.com/appforest_uf/f1678949...,92
87,https://att-104164.weeblysite.com/,34
91,http://tkmowpikuk.owl4fsrch.club/vnafvra97w/?q...,64
92,https://a04854e8-0cca-4b74-80e6-aacb14a4a5f7.i...,55


In [41]:
# Check the unusual URL length differences
length_difference = (
    df["URL"].astype(str).str.len() - df["URLLength"]
)

unusual_rows = df.loc[
    length_difference.isin([4, 35]),
    ["URL", "URLLength", "label"]
].copy()

unusual_rows["Actual_Length"] = unusual_rows["URL"].astype(str).str.len()
unusual_rows["Difference"] = (
    unusual_rows["Actual_Length"] - unusual_rows["URLLength"]
)

display(unusual_rows)

,URL,URLLength,label,Actual_Length,Difference
127839,https://22017-5502.s3.webspace.re/privatkunden...,128,0,132,4
158380,https://vinted.6545657.xyz/p5ci5kt9Ã¢â‚¬â€¹Ã¢â...,65,0,100,35


In [42]:
for idx in [127839, 158380]:
    url = str(df.loc[idx, "URL"])

    print("\nRow:", idx)
    print("Full URL:", repr(url))
    print("Actual string length:", len(url))
    print("Dataset URLLength:", df.loc[idx, "URLLength"])
    print("Difference:", len(url) - df.loc[idx, "URLLength"])

    print("\nNon-ASCII characters:")
    for pos, char in enumerate(url):
        if ord(char) > 127:
            print(
                "Position:", pos,
                "| Character:", repr(char),
                "| Unicode:", f"U+{ord(char):04X}"
            )


Row: 127839
Full URL: 'https://22017-5502.s3.webspace.re/privatkunden/Ã£Å“berblick/sicherheit/notfallhilfe/firmenkunden/geschÃ£Â¤ftskunden/kontakt/clients/'
Actual string length: 132
Dataset URLLength: 128
Difference: 4

Non-ASCII characters:
Position: 47 | Character: 'Ã' | Unicode: U+00C3
Position: 48 | Character: '£' | Unicode: U+00A3
Position: 49 | Character: 'Å' | Unicode: U+00C5
Position: 50 | Character: '“' | Unicode: U+201C
Position: 102 | Character: 'Ã' | Unicode: U+00C3
Position: 103 | Character: '£' | Unicode: U+00A3
Position: 104 | Character: 'Â' | Unicode: U+00C2
Position: 105 | Character: '¤' | Unicode: U+00A4

Row: 158380
Full URL: 'https://vinted.6545657.xyz/p5ci5kt9Ã¢â‚¬â€¹Ã¢â‚¬â€¹Ã¢â‚¬â€¹Ã¢â‚¬â€¹Ã¢â‚¬â€¹Ã¢â‚¬â€¹Ã¢â‚¬â€¹/tsidom/7'
Actual string length: 100
Dataset URLLength: 65
Difference: 35

Non-ASCII characters:
Position: 35 | Character: 'Ã' | Unicode: U+00C3
Position: 36 | Character: '¢' | Unicode: U+00A2
Position: 37 | Character: 'â' | Unicode: U+00E2
Position: 38

In [43]:
length_check = pd.DataFrame({
    "Actual_Length": df["URL"].astype(str).str.len(),
    "Dataset_Length": df["URLLength"],
    "Label": df["label"]
})

length_check["Difference"] = (
    length_check["Actual_Length"] - length_check["Dataset_Length"]
)

print("Difference distribution by label:")
display(
    length_check.groupby(["Label", "Difference"])
    .size()
    .unstack(fill_value=0)
)

print("\nMean actual and stored length by label:")
display(
    length_check.groupby("Label")[
        ["Actual_Length", "Dataset_Length", "Difference"]
    ].mean().round(2)
)

Difference distribution by label:


Difference,0,1,4,35
Label,,,,
0,48644,52299,1,1
1,0,134850,0,0



Mean actual and stored length by label:


,Actual_Length,Dataset_Length,Difference
Label,,,
0,46.24,45.72,0.52
1,27.23,26.23,1.00


In [44]:
legit_examples = df.loc[
    (df["label"] == 1) &
    ((df["URL"].astype(str).str.len() - df["URLLength"]) == 1),
    ["URL", "URLLength"]
].head(5).copy()

legit_examples["Actual_Length"] = (
    legit_examples["URL"].astype(str).str.len()
)

legit_examples["Actual_minus_Dataset"] = (
    legit_examples["Actual_Length"] - legit_examples["URLLength"]
)

display(legit_examples)

,URL,URLLength,Actual_Length,Actual_minus_Dataset
0,https://www.southbankmosaics.com,31,32,1
1,https://www.uni-mainz.de,23,24,1
2,https://www.voicefmradio.co.uk,29,30,1
3,https://www.sfnmjournal.com,26,27,1
4,https://www.rewildingargentina.org,33,34,1


In [45]:
# Compare actual and stored URL lengths by label
actual_length = df["URL"].astype(str).str.len()
difference = actual_length - df["URLLength"]

print("Phishing URLs (label 0):")
print(difference[df["label"] == 0].value_counts().sort_index())

print("\nLegitimate URLs (label 1):")
print(difference[df["label"] == 1].value_counts().sort_index())

print("\nCheck URLLength data type:")
print(df["URLLength"].dtype)

Phishing URLs (label 0):
0     48644
1     52299
4         1
35        1
Name: count, dtype: int64

Legitimate URLs (label 1):
1    134850
Name: count, dtype: int64

Check URLLength data type:
int64
